# Exploratory Data Analysis — World Cup 2026 Predictor

Quick look at the historical match data and engineered features before modeling.
Run `python scripts/download_sample_data.py` and `python -m src.data.build_features` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

features = pd.read_csv('../data/processed/match_features.csv', parse_dates=['date'])
features.head()

## Outcome distribution

Home advantage is real but draws are common enough that a model has to take them seriously rather than treating football as a binary outcome.

In [ ]:
outcome_labels = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}
counts = features['result'].map(outcome_labels).value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', ax=ax, color=['#d62728', '#7f7f7f', '#2ca02c'])
ax.set_title('Match Outcome Distribution')
ax.set_ylabel('Number of Matches')
plt.tight_layout()
plt.show()

print(counts / counts.sum())

## ELO differential vs outcome

This is the core signal in the model. The wider the rating gap, the more lopsided the outcome distribution should be.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for outcome, label, color in [(2, 'Home Win', '#2ca02c'), (1, 'Draw', '#7f7f7f'), (0, 'Away Win', '#d62728')]:
    subset = features[features['result'] == outcome]
    sns.kdeplot(subset['elo_diff'], label=label, color=color, ax=ax)

ax.set_title('ELO Rating Differential by Match Outcome')
ax.set_xlabel('Home ELO - Away ELO')
ax.legend()
plt.tight_layout()
plt.show()

## Calibration check: does win probability roughly match observed win rate?

Bucket matches by ELO differential and check whether home teams actually won at the rate the rating gap implies. This is a sanity check before trusting the model's probabilities.

In [ ]:
features['elo_bucket'] = pd.cut(features['elo_diff'], bins=[-1000, -150, -50, 50, 150, 1000],
                                  labels=['Strong Underdog', 'Slight Underdog', 'Even', 'Slight Favorite', 'Strong Favorite'])

home_win_rate_by_bucket = features.groupby('elo_bucket')['result'].apply(lambda x: (x == 2).mean())
print(home_win_rate_by_bucket)

fig, ax = plt.subplots(figsize=(7, 4))
home_win_rate_by_bucket.plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Home Win Rate by ELO Differential Bucket')
ax.set_ylabel('Home Win Rate')
plt.tight_layout()
plt.show()

## Takeaways

- Home win rate increases monotonically with ELO advantage, which is the expected and necessary pattern for the rating system to be useful as a model feature.
- Draws make up a meaningful share of outcomes (roughly 25%) even in lopsided matchups, which is why this is framed as a 3-class problem rather than a binary win/loss model.
- The 'Even' bucket should show close to a 50% home win rate (after accounting for home advantage) — deviations here would suggest a bug in the ELO update logic.